# CrossFuse v5 -- Stage A: FakeAVCeleb Extraction

Caches face crops + waveform + BOTH audio-forgery MFCC variants (with and
without the leading-silence skip) + full-clip log-mel, for every clip in the
training plan. Run once, then **Save Version**, publish `/kaggle/working/crops_v5`
as a Kaggle Dataset, and attach it to every downstream notebook -- do not
re-run this cell from NB-B/C/D.

Budget: ~30-40 min for ~1,500 clips on a 2xT4 session (extraction is the one
stage that genuinely benefits from 2 GPUs, via the ThreadPoolExecutor below).

Attach as input: the FakeAVCeleb Kaggle dataset, and the `crossfuse-v5-lib`
dataset containing `crossfuse_v5.py`.


In [ ]:
!pip install -q --no-deps facenet-pytorch
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, hashlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from crossfuse_v5 import (
    CONFIG, make_rng, ffmpeg_extract_wav, mfcc_from_waveform, logmel_full,
    build_identity_groups, build_detector, extract_face_crops_contiguous,
    get_fakeavceleb_label, get_fakeavceleb_category, category_to_modality_labels,
    make_sample_key, CATEGORY_SHORT,
)

N_DEVICES = max(1, torch.cuda.device_count())
DEVICE_IDS = [f"cuda:{i}" for i in range(N_DEVICES)] if torch.cuda.is_available() else ["cpu"]
print(f"Extraction devices: {DEVICE_IDS}")

CROP_DIR = "/kaggle/working/crops_v5"
os.makedirs(CROP_DIR, exist_ok=True)
manifest_path = os.path.join(CROP_DIR, "manifest.csv")


In [ ]:
# ---- 1. Locate the FakeAVCeleb pool, resolve identities, sample per-category ----
all_videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
if len(all_videos) == 0:
    all_videos = sorted(glob.glob("/kaggle/input/**/*.avi", recursive=True))

fav_videos = [v for v in all_videos
              if not any(t in v.lower() for t in
                        ("dfdc", "deepfake-detection", "celeb-df", "celebdf",
                         "asvspoof", "ff-c23", "faceforensics", "in-the-wild"))]

real_videos, fake_videos = [], []
for v in fav_videos:
    (real_videos if get_fakeavceleb_label(v) == 0 else fake_videos).append(v)
print(f"Raw candidate pool: {len(real_videos)} real | {len(fake_videos)} fake")

print("Resolving identity groups (union-find over id-tokens in path + filename)...")
identity_groups = build_identity_groups(fav_videos)
print(f"  {len(fav_videos)} clips -> {len(set(identity_groups.values()))} identity groups")

rng_cat = make_rng("category_sample")
rng_cat.shuffle(real_videos)
rng_cat.shuffle(fake_videos)

selected_reals = real_videos  # the real pool is small; use all of it

fakes_by_cat = defaultdict(list)
for v in fake_videos:
    fakes_by_cat[get_fakeavceleb_category(v)].append(v)
for cat in fakes_by_cat:
    rng_cat.shuffle(fakes_by_cat[cat])

selected_fakes = []
selected_fakes += fakes_by_cat.get("RealVideo-FakeAudio", [])                      # keep ALL
selected_fakes += fakes_by_cat.get("FakeVideo-RealAudio", [])[:CONFIG["CAP_FVRA"]]
selected_fakes += fakes_by_cat.get("FakeVideo-FakeAudio", [])[:CONFIG["CAP_FVFA"]]
rng_cat.shuffle(selected_fakes)

clip_list = selected_reals + selected_fakes
plan_counts = defaultdict(int)
for v in clip_list:
    plan_counts[get_fakeavceleb_category(v)] += 1
print(f"\nClips to cache: {len(clip_list)}  |  per-category -> {dict(plan_counts)}")
est_gb = len(clip_list) * CONFIG["CACHED_FRAMES"] * (CONFIG["CROP_SIZE"] ** 2) * 3 / 1e9
print(f"estimated crop storage: {est_gb:.2f} GB  (Kaggle notebook-output limit is 20 GB)")

jobs = []
for v in clip_list:
    cat = get_fakeavceleb_category(v)
    jobs.append(dict(video_path=v, category=cat, identity_group=identity_groups[v],
                     key=make_sample_key(cat, identity_groups[v], v)))
_keys = [j["key"] for j in jobs]
assert len(_keys) == len(set(_keys)), (
    f"Key collision: {len(_keys) - len(set(_keys))} duplicates -- this assert exists "
    "specifically to catch silently-dropped clips.")
print(f"Extraction jobs queued: {len(jobs)} (all keys unique)")


In [ ]:
# ---- 2. Per-clip extraction: crops, waveform, BOTH MFCC variants, full log-mel ----
def process_job(job, device_str, mtcnn_local):
    key = job["key"]
    crop_path = os.path.join(CROP_DIR, f"{key}.npy")
    meta_path = os.path.join(CROP_DIR, f"{key}_meta.npz")
    if os.path.exists(crop_path) and os.path.exists(meta_path):
        meta = np.load(meta_path)
        return job, "cached", dict(n_valid=int(meta["valid"].sum()),
                                   fps=float(1.0 / (meta["timestamps"][1] - meta["timestamps"][0]))
                                   if len(meta["timestamps"]) > 1 else 25.0)

    rng = make_rng(key)
    try:
        crops, valid, fps, start_frame, timestamps = extract_face_crops_contiguous(
            job["video_path"], mtcnn_local, rng=rng)
    except Exception as e:
        return job, f"visual_error:{type(e).__name__}", None
    if crops is None or int(valid.sum()) == 0:
        return job, "no_faces", None

    try:
        y, sr = ffmpeg_extract_wav(job["video_path"])
    except Exception as e:
        return job, f"audio_error:{type(e).__name__}", None
    if y is None or len(y) < sr // 10:
        return job, "audio_too_short", None

    mfcc_full = mfcc_from_waveform(y, sr, lead_skip=0.0)
    mfcc_trim = mfcc_from_waveform(y, sr, lead_skip=CONFIG["AUDIO_LEAD_SKIP"])
    mel_full = logmel_full(y, sr)

    np.save(crop_path, crops)
    np.savez(meta_path, valid=valid, timestamps=timestamps,
             mfcc_full=mfcc_full, mfcc_trim=mfcc_trim, mel=mel_full)
    return job, "ok", dict(n_valid=int(valid.sum()), fps=float(fps))


detectors = {d: build_detector(d) for d in DEVICE_IDS}
# A thread pool does not pin a job to a worker. Protect each shared detector.
import threading
detector_locks = {d: threading.Lock() for d in DEVICE_IDS}
_process_job_unlocked = process_job
def process_job(job, *args):
    detector = args[-1]
    device_id = next(d for d, instance in detectors.items() if instance is detector)
    with detector_locks[device_id]:
        return _process_job_unlocked(job, *args)
results_log = defaultdict(int)
manifest_rows = []

with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futures = [pool.submit(process_job, job, DEVICE_IDS[i % len(DEVICE_IDS)],
                           detectors[DEVICE_IDS[i % len(DEVICE_IDS)]])
              for i, job in enumerate(jobs)]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Caching (v5, multi-GPU)"):
        job, status, meta = fut.result()
        results_log[status] += 1
        if status in ("ok", "cached"):
            v_lab, a_lab = category_to_modality_labels(job["category"])
            manifest_rows.append(dict(
                key=job["key"], path=job["video_path"], category=job["category"],
                identity_group=job["identity_group"], video_label=v_lab, audio_label=a_lab,
                n_valid_faces=meta["n_valid"], fps=round(meta["fps"], 3)))

print(f"\nExtraction outcomes: {dict(results_log)}")


In [ ]:
# ---- 3. Manifest write + integrity check ----
manifest_rows.sort(key=lambda r: r["key"])
with open(manifest_path, "w", newline="") as f:
    fieldnames = ["key", "path", "category", "identity_group", "video_label",
                  "audio_label", "n_valid_faces", "fps"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

on_disk = {os.path.splitext(os.path.basename(p))[0]
           for p in glob.glob(os.path.join(CROP_DIR, "*.npy"))}
manifest_keys = {r["key"] for r in manifest_rows}
assert len(manifest_rows) == len(manifest_keys), "Duplicate keys in manifest!"
missing = manifest_keys - on_disk
assert not missing, f"{len(missing)} manifest rows have no .npy on disk: {list(missing)[:5]}"
for r in manifest_rows:
    assert os.path.exists(os.path.join(CROP_DIR, r["key"] + "_meta.npz")), \
        f"Missing meta for {r['key']}"

total_bytes = sum(os.path.getsize(p) for p in glob.glob(os.path.join(CROP_DIR, "*.npy")))
cat_final, vlab_final, alab_final = defaultdict(int), defaultdict(int), defaultdict(int)
for r in manifest_rows:
    cat_final[r["category"]] += 1
    vlab_final[r["video_label"]] += 1
    alab_final[r["audio_label"]] += 1

print(f"Manifest OK -> {len(manifest_rows)} rows, {len(on_disk)} crop files on disk.")
print(f"Per-category: {dict(cat_final)}")
print(f"Video real/fake: {vlab_final[0]}/{vlab_final[1]} | "
      f"Audio real/fake: {alab_final[0]}/{alab_final[1]}")
print(f"Total cache size: {total_bytes / 1e9:.2f} GB")
if total_bytes / 1e9 > 19.0:
    print("[!] Approaching Kaggle's 20 GB output limit -- lower CACHED_FRAMES or CROP_SIZE.")

if manifest_rows:
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
    for ax, r in zip(axes, manifest_rows[:5]):
        arr = np.load(os.path.join(CROP_DIR, r["key"] + ".npy"), mmap_mode="r")
        ax.imshow(arr[0]); ax.axis("off")
        ax.set_title(f"{r['category'][:12]}\nvalid {r['n_valid_faces']}", fontsize=8)
    plt.suptitle("Cached crop spot-check", fontsize=11)
    plt.tight_layout(); plt.show()

print(f"\nManifest written to: {manifest_path}")
print("Stage A complete. Save Version, publish /kaggle/working/crops_v5 as a "
      "Kaggle Dataset (e.g. 'crops-v5'), attach it to NB-B/C/D.")
